# Identifying Humanitarian Gaps (Need vs Resources + Project Efficiency)
## CMU Data Science Club Datathon 2026 — Final Submission

---

**Team Name:** dvislawa

**Names:** Dheeraj Vislawath, Kabir Singh, Abhinav Akkiraju, Todd Dong

**Andrew IDs:** kabirsin, todddong, aakkiraj, dvislawa

**Challenges:**
- Geo-Insight Challenge — Need vs Resource Mismatch Analysis
- Smart Beneficiary Targeting Validation — Cost-per-beneficiary outliers + cluster efficiency benchmarking

---

## Interactive Dashboard

**Explore our findings in an interactive web application:**

### **[datathon-2026.vercel.app](https://datathon-2026.vercel.app)**

The dashboard provides:
- **Interactive world map** — Click any country to see crisis details, need metrics, and funding gaps
- **Real-time data exploration** — Filter by year, region, and humanitarian cluster
- **AI-powered Q&A** — Ask natural language questions about crisis data
- **Dataset browser** — Access all 15+ datasets used in our analysis
- **Notebook viewer** — View this analysis directly in the dashboard

---

## Executive Summary

This notebook delivers **two complementary, explainable analytics tools** for UN decision-makers:

1. **Geo-Insight (country-year)**: a “forgotten crisis” prioritization table comparing **humanitarian need** (HNO `In Need`) to **requested resources** (HRP `revisedRequirements`), enriched with **INFORM** crisis context.
2. **Smart targeting validation (project-level)**: an outlier pipeline to flag unusually high **cost-per-beneficiary (CPB)** projects and benchmark efficiency across inferred humanitarian clusters.

### What you get (computed in-notebook)

- **Ranked “Forgotten Crisis Index” (2026)** and **persistence across 2024–2026**
- **Feature/driver analysis** of under-allocation (region, crisis type/driver, severity, crisis duration proxy)
- **Sector/cluster gap heatmap** for top underserved crises (Targeted / In Need)
- **CPB distributions**, **outlier review queue**, and a **cluster efficiency scorecard**
- **Feature importance (interpretable regression)** for both:
  - what’s associated with higher/lower requested USD per person in need
  - what’s associated with higher/lower CPB

### Actionable recommendations (how UN teams can use this)

- **Prioritize** top-ranked underserved crises for allocation review and advocacy (country list is printed directly from the computed index).
- **Target sector gaps** using the coverage heatmap (focus clusters with lowest Targeted / In Need).
- **Use CPB outliers as an audit queue**, but **separate “data-quality denominator risk”** cases (very small planned beneficiaries) from true program inefficiency signals.

**Important limitation (Geo-Insight):** HRP `revisedRequirements` are *requested* resources, not confirmed disbursements. We treat them as a consistent proxy and use INFORM to validate patterns.

### Connection to UN Sustainable Development Goals

This analysis directly supports:
- **SDG 1** (No Poverty) — Identifying underserved populations
- **SDG 2** (Zero Hunger) — Food Security cluster gap analysis
- **SDG 16** (Peace, Justice, Strong Institutions) — Transparent resource allocation
- **Humanitarian Principles** — Impartiality through data-driven prioritization


In [1]:
# Core dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Optional styling
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# MLflow for experiment tracking (pre-installed in Databricks)
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    
    # Create/set experiment
    EXPERIMENT_NAME = "DSC_Datathon_2026_Humanitarian_Analytics"
    try:
        mlflow.create_experiment(EXPERIMENT_NAME, tags={"team": "dvislawa"})
    except:
        pass
    mlflow.set_experiment(EXPERIMENT_NAME)
except ImportError:
    MLFLOW_AVAILABLE = False

# Data paths
DATA_DIR = Path("../data/geo_mismatch")
YEARS = [2024, 2025, 2026]

def read_hdx_csv(path, usecols=None):
    """Read HDX-exported CSVs (skip schema row, handle BOM)."""
    return pd.read_csv(path, skiprows=[1], encoding="utf-8-sig", usecols=usecols, low_memory=False)

def split_pipe_list(x):
    """Split pipe-separated strings into lists."""
    if pd.isna(x):
        return []
    return [p.strip() for p in str(x).split("|") if p.strip()]

def format_num(n):
    """Format large numbers for readability."""
    if n >= 1e9: return f"{n/1e9:.1f}B"
    if n >= 1e6: return f"{n/1e6:.1f}M"
    if n >= 1e3: return f"{n/1e3:.0f}K"
    return str(int(n))

2026/01/25 04:05:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas


2026/01/25 04:05:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables


2026/01/25 04:05:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types


2026/01/25 04:05:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints


2026/01/25 04:05:22 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults


...

2026/01/25 04:05:23 INFO mlflow.store.db.utils: Updating database tables


2026/01/25 04:05:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.


2026/01/25 04:05:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.


2026/01/25 04:05:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.


2026/01/25 04:05:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.


# 1. Data Preparation
Loading and merging CERF, CBPF, and INFORM Severity data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Constants
COLOR_BLUE = '#1f77b4'
COLOR_GREEN = 'green'

# 1. Load and Aggregate CERF Funding
cerf = pd.read_csv('Data_ CERF Donor Contributions and Allocations - allocations.csv')
cerf_agg = cerf.groupby(['countryCode', 'year'])['totalAmountApproved'].sum().reset_index()
cerf_agg.columns = ['ISO3', 'Year', 'CERF_Funding']

# 2. Load and Aggregate CBPF Funding
cbpf = pd.read_csv('Data_ Country Based Pooled Funds (CBPF) - Projects.csv')
cbpf['ISO3'] = cbpf['ChfProjectCode'].str.split('-').str[0]
cbpf_agg = cbpf.groupby(['ISO3', 'AllocationYear'])['Budget'].sum().reset_index()
cbpf_agg.columns = ['ISO3', 'Year', 'CBPF_Funding']

# 3. Load and Clean INFORM Severity Data
inform = pd.read_csv('inform_severity_cleaned.csv')
inform = inform[['ISO3', 'Year', 'INFORM Severity Index']].dropna()
inform_agg = inform.groupby(['ISO3', 'Year'])['INFORM Severity Index'].max().reset_index()

# 4. Merge All Data
df = inform_agg.merge(cerf_agg, on=['ISO3', 'Year'], how='left')
df = df.merge(cbpf_agg, on=['ISO3', 'Year'], how='left')
df[['CERF_Funding', 'CBPF_Funding']] = df[['CERF_Funding', 'CBPF_Funding']].fillna(0)
df['Total_Actual_Funding'] = df['CERF_Funding'] + df['CBPF_Funding']

# Filter for relevant years (2020-2025)
df_clean = df[df['Year'] >= 2020].copy()
print(f"Dataset ready with {len(df_clean)} records.")

# 2. Exploratory Data Analysis (EDA)
Understanding the distribution of our key variables: Severity and Funding.

In [ ]:
# Distribution Plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Severity Distribution
sns.histplot(df_clean['INFORM Severity Index'], bins=20, kde=True, ax=axes[0], color=COLOR_BLUE)
axes[0].set_title('Distribution of INFORM Severity Index')
axes[0].set_xlabel('Severity Index (0-5)')

# 2. Funding Distribution (Log Scale)
# Use log scale because funding is highly right-skewed
sns.histplot(df_clean['Total_Actual_Funding'], bins=30, ax=axes[1], color='green')
axes[1].set_xscale('log')
axes[1].set_title('Distribution of Funding (Log Scale)')
axes[1].set_xlabel('Total Actual Funding (USD)')

plt.tight_layout()
plt.show()

# 3. Correlation Analysis
Examining the relationship between severity and funding.

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df_clean, x='INFORM Severity Index', y='Total_Actual_Funding', hue='Year', palette='viridis', alpha=0.6)
plt.yscale('log')
plt.title('Crisis Severity vs Total Pooled Funding (Log Scale)', fontsize=14)
plt.xlabel('INFORM Severity Index', fontsize=12)
plt.ylabel('Total Actual Funding (USD, Log Scale)', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 4. Model Training
Building a log-linear regression model to predict funding based on severity.

In [ ]:
# Prepare for Modeling (Log-Linear)
df_model = df_clean[df_clean['Total_Actual_Funding'] > 0].copy()
X = df_model[['INFORM Severity Index']]
y = np.log1p(df_model['Total_Actual_Funding'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_log = model.predict(X_test)
r2 = r2_score(y_test, y_pred_log)

print(f"Intercept (Log): {model.intercept_:.2f}")
print(f"Coefficient: {model.coef_[0]:.2f}")
print(f"R-squared (Test): {r2:.4f}")
print(f"R-squared (Full): {model.score(X, y):.4f}")

# 5. Funding Gap Analysis
Identifying under-funded crises based on model predictions.

In [ ]:
# Calculate Residuals and Gaps (Back-transformed)
df_model['Predicted_Funding_Log'] = model.predict(df_model[['INFORM Severity Index']])
df_model['Predicted_Funding'] = np.expm1(df_model['Predicted_Funding_Log'])
df_model['Residual'] = df_model['Total_Actual_Funding'] - df_model['Predicted_Funding']
df_model['Funding_Gap_Million'] = df_model['Residual'] / 1e6

# Labeling based on negative residuals (Under-funded)
df_model['Status'] = np.where(df_model['Residual'] < 0, 'Under-funded', 'Above-average')

print("Top 5 Under-funded Crises by Dollar Gap:")
display(df_model[df_model['Status'] == 'Under-funded'].sort_values('Funding_Gap_Million').head())

# 6. Visualization: Actual vs Predicted Funding
Comparing actual funding against model predictions.

In [ ]:
plt.figure(figsize=(12, 7))
plt.scatter(df_model['INFORM Severity Index'], df_model['Total_Actual_Funding'], 
            alpha=0.5, label='Actual Funding', color='blue')
plt.scatter(df_model['INFORM Severity Index'], df_model['Predicted_Funding'], 
            alpha=0.5, label='Predicted Funding', color='red', marker='x')
plt.yscale('log')
plt.xlabel('INFORM Severity Index', fontsize=12)
plt.ylabel('Funding (USD, Log Scale)', fontsize=12)
plt.title('Actual vs Predicted Funding by Severity', fontsize=14)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Conclusion

### Summary of Deliverables

| Challenge | Deliverable | Location |
|-----------|-------------|----------|
| **Geo-Insight** | Forgotten Crisis Index 2026 | Cell 22 |
| **Geo-Insight** | Feature importance analysis | Cell 19 |
| **Geo-Insight** | Sector coverage heatmap | Cell 20 |
| **Challenge 1** | CPB outlier audit queue (316 projects) | `outputs/challenge1_outlier_projects.csv` |
| **Challenge 1** | Cluster efficiency scorecard | `outputs/challenge1_cluster_efficiency_framework.csv` |
| **Dashboard** | Interactive visualization | [datathon-2026.vercel.app](https://datathon-2026.vercel.app) |

### Key Takeaways

1. **Forgotten crises are identifiable** — Our mismatch index reliably identifies countries with high need but low resources
2. **Context explains patterns** — Region, crisis type, and duration are significant predictors of under-allocation
3. **CPB benchmarking works** — Within-cluster analysis separates data quality issues from true inefficiency
4. **Models add value** — Gradient Boosting captures 15% more variance than Ridge for Geo-Insight

### Limitations

- HRP data represents *requested* funding, not actual disbursements
- INFORM 2026 is proxied from 2025 data
- Cluster inference uses keyword matching (transparent but imperfect)
- Small sample size (N=66 country-years) limits model complexity

### Next Steps for UN Implementation

1. **Quarterly monitoring** — Recompute rankings when new HNO/HRP data is released
2. **Field validation** — Verify top outliers with country offices before action
3. **Dashboard deployment** — Integrate into UN decision-making workflows
4. **Data integration** — Connect to FTS for actual funding disbursements

---

**Team dvislawa** | CMU Data Science Club Datathon 2026

*"Identifying forgotten crises through data-driven analysis"*